In [ ]:

import pandas as pd
import numpy as np
import diptest
import os

from sklearn.mixture import GaussianMixture

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.csv"


if os.path.exists(INPUT_CSV):
    df = pd.read_csv(INPUT_CSV)
    print("CSV loaded successfully")
else:
    print("File not found:")
    print(INPUT_CSV)

print("\n")
print("="*80)
print(" GBM THICKNESS MULTIMODALITY ANALYSIS ")
print("="*80)

results = []

#Analysis per patient

for patient, data in df.groupby("patient_id"):
        thickness = (
        data["median_thickness_nm"]
        .dropna()
        .values
    )
    n_samples = len(thickness)

    print("\n")
    print("-"*80)
    print(f"Patient : {patient}")
    print(f"Membrane samples : {n_samples}")

    if n_samples < 5:
        print("Not enough membranes for statistical analysis")
        continue

  
#Hartigan's dip test 
    dip_value, dip_p = diptest.diptest(thickness)
    if dip_p < 0.05:
        dip_result = "Multimodal"

    else:
        dip_result = "Unimodal"

  
#Gaussian Mixture Model 
    X = thickness.reshape(-1,1)
    models = {}
    bic = {}
    aic = {}

    for components in [1,2,3]:
        model = GaussianMixture(
            n_components=components,
            random_state=42,
            n_init=20
        )
        model.fit(X)
        models[components] = model
        bic[components] = model.bic(X)
        aic[components] = model.aic(X)

    best_components = min(
        bic,
        key=bic.get
    )
    best_model = models[best_components]

    peaks = sorted(
        best_model.means_.flatten()
    )
    while len(peaks) < 3:

        peaks.append(np.nan)
  
    results.append({

        "Patient_ID": patient,
        "Number_of_membranes": n_samples,
        "Dip_statistic":
            round(dip_value,5),

        "Dip_p_value":
            round(dip_p,5),

        "Dip_test_result":
            dip_result,

        "Best_GMM_components":
            best_components,

        "BIC":
            round(
                bic[best_components],
                3
            ),

        "AIC":
            round(
                aic[best_components],
                3
            ),

        "Peak_1_nm":
            round(peaks[0],2),

        "Peak_2_nm":
            round(peaks[1],2)
            if not np.isnan(peaks[1])
            else None,

        "Peak_3_nm":
            round(peaks[2],2)
            if not np.isnan(peaks[2])
            else None
    })

    print(
        f"Dip test : {dip_result}"
    )
    print(
        f"Best GMM : {best_components} component(s)"
    )
    print(
        f"Peaks : {peaks}"
    )

summary = pd.DataFrame(results)

summary.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n")
print("="*80)
print("Analysis completed")
print("="*80)
print(summary)
print(
    "\nSaved file:"
)
print(
    OUTPUT_CSV
)

CSV loaded successfully


 GBM THICKNESS MULTIMODALITY ANALYSIS 


--------------------------------------------------------------------------------
Patient : 01-24
Membrane samples : 19
Dip test : Unimodal
Best GMM : 3 component(s)
Peaks : [np.float64(170.90564610527923), np.float64(372.42173332897664), np.float64(480.6154006222281)]


--------------------------------------------------------------------------------
Patient : 02-24
Membrane samples : 41
Dip test : Unimodal
Best GMM : 3 component(s)
Peaks : [np.float64(241.72031603248845), np.float64(785.1336126935767), np.float64(1339.9615827150287)]


--------------------------------------------------------------------------------
Patient : 03-24
Membrane samples : 22
Dip test : Unimodal
Best GMM : 2 component(s)
Peaks : [np.float64(196.40105228935488), np.float64(576.6165804143551), nan]


--------------------------------------------------------------------------------
Patient : 04-23
Membrane samples : 51
Dip test : Unimodal
Best GMM